OpenAQ Air Quality Data Engineering: raw ingestion, data quality, staging, mart, and analytic queries in Snowflake
*Co-authored with CoCo*

# OpenAQ Air Quality Data Mart
Complete data engineering pipeline: raw ingestion → data quality → staging → data mart → analytic queries

## Step 1: Create Database and Schemas

In [ ]:
%%sql -r create_db
CREATE OR REPLACE DATABASE OPENAQ;

In [ ]:
%%sql -r create_schemas
USE DATABASE OPENAQ;

CREATE OR REPLACE SCHEMA RAW
    COMMENT = 'Raw ingestion layer for OpenAQ data';

CREATE OR REPLACE SCHEMA STAGING
    COMMENT = 'Cleaned and validated data';

CREATE OR REPLACE SCHEMA MART
    COMMENT = 'Data mart optimized for analytic queries';

CREATE OR REPLACE SCHEMA DATA_QUALITY
    COMMENT = 'Data quality reporting and rejected records';

## Step 2: Create Raw Table

In [ ]:
%%sql -r create_raw_table
USE SCHEMA OPENAQ.RAW;

CREATE OR REPLACE TABLE AIR_QUALITY_MEASUREMENTS (
    location_id     INTEGER,
    sensors_id      INTEGER,
    location        VARCHAR(500),
    datetime        TIMESTAMP_TZ,
    lat             FLOAT,
    lon             FLOAT,
    parameter       VARCHAR(50),
    units           VARCHAR(50),
    value           FLOAT,
    ingested_at     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Raw air quality measurements from OpenAQ S3 bucket';

## Step 3: File Formats and Stages

In [ ]:
%%sql -r create_file_format
CREATE OR REPLACE FILE FORMAT OPENAQ.RAW.CSV_FORMAT
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    NULL_IF = ('', 'NA', 'null')
    EMPTY_FIELD_AS_NULL = TRUE
    ENCODING = 'UTF8';

In [ ]:
%%sql -r create_stages
-- External stage pointing to OpenAQ S3 public bucket
CREATE OR REPLACE STAGE OPENAQ.RAW.OPENAQ_S3_STAGE
    URL = 's3://openaq-fetches/'
    FILE_FORMAT = OPENAQ.RAW.CSV_FORMAT;

-- Internal stage for local file uploads
CREATE OR REPLACE STAGE OPENAQ.RAW.INTERNAL_STAGE
    FILE_FORMAT = OPENAQ.RAW.CSV_FORMAT;

## Step 4: Load Sample Data from Workspace CSV

In [ ]:
%%sql -r copy_to_stage
-- Ensure stage exists before copying
CREATE STAGE IF NOT EXISTS OPENAQ.RAW.INTERNAL_STAGE
    FILE_FORMAT = OPENAQ.RAW.CSV_FORMAT;

-- Copy workspace file to internal stage
COPY FILES INTO @OPENAQ.RAW.INTERNAL_STAGE
FROM 'snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live'
FILES=('location-2178-20220503.csv');

In [ ]:
%%sql -r load_data
-- Load from stage into raw table
COPY INTO OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS (
    location_id, sensors_id, location, datetime, lat, lon, parameter, units, value
)
FROM @OPENAQ.RAW.INTERNAL_STAGE/location-2178-20220503.csv
FILE_FORMAT = (FORMAT_NAME = OPENAQ.RAW.CSV_FORMAT)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r verify_load
-- Verify load
SELECT COUNT(*) AS raw_record_count FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS;

## Step 5: Data Quality - Rejected Records

In [ ]:
%%sql -r create_dq_table
USE SCHEMA OPENAQ.DATA_QUALITY;

CREATE OR REPLACE TABLE REJECTED_RECORDS (
    location_id     INTEGER,
    sensors_id      INTEGER,
    location        VARCHAR(500),
    datetime        TIMESTAMP_TZ,
    lat             FLOAT,
    lon             FLOAT,
    parameter       VARCHAR(50),
    units           VARCHAR(50),
    value           FLOAT,
    rejection_reason VARCHAR(200),
    rejected_at     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Records excluded from the data mart due to quality issues';

In [ ]:
%%sql -r reject_nulls
-- Reject records with NULL values in critical fields
INSERT INTO OPENAQ.DATA_QUALITY.REJECTED_RECORDS
    (location_id, sensors_id, location, datetime, lat, lon, parameter, units, value, rejection_reason)
SELECT location_id, sensors_id, location, datetime, lat, lon, parameter, units, value,
    CASE
        WHEN value IS NULL THEN 'NULL measurement value'
        WHEN datetime IS NULL THEN 'NULL datetime'
        WHEN location IS NULL OR location = '' THEN 'NULL or empty location'
        WHEN parameter IS NULL OR parameter = '' THEN 'NULL or empty parameter'
        WHEN lat IS NULL OR lon IS NULL THEN 'NULL coordinates'
    END AS rejection_reason
FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS
WHERE value IS NULL
   OR datetime IS NULL
   OR location IS NULL OR location = ''
   OR parameter IS NULL OR parameter = ''
   OR lat IS NULL OR lon IS NULL;

In [ ]:
%%sql -r reject_negatives
-- Reject negative values (physically impossible for concentrations)
INSERT INTO OPENAQ.DATA_QUALITY.REJECTED_RECORDS
    (location_id, sensors_id, location, datetime, lat, lon, parameter, units, value, rejection_reason)
SELECT location_id, sensors_id, location, datetime, lat, lon, parameter, units, value,
    'Negative measurement value' AS rejection_reason
FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS
WHERE value < 0;

In [ ]:
%%sql -r reject_outliers
-- Reject extreme outliers (>5 standard deviations from mean per parameter)
INSERT INTO OPENAQ.DATA_QUALITY.REJECTED_RECORDS
    (location_id, sensors_id, location, datetime, lat, lon, parameter, units, value, rejection_reason)
SELECT r.location_id, r.sensors_id, r.location, r.datetime, r.lat, r.lon, r.parameter, r.units, r.value,
    'Extreme outlier (>5 std dev from mean)' AS rejection_reason
FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS r
JOIN (
    SELECT parameter, AVG(value) AS avg_val, STDDEV(value) AS std_val
    FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS
    WHERE value IS NOT NULL AND value >= 0
    GROUP BY parameter
) stats ON r.parameter = stats.parameter
WHERE r.value > (stats.avg_val + 5 * stats.std_val)
  AND stats.std_val > 0;

In [ ]:
%%sql -r dq_summary
-- Data quality summary report
SELECT rejection_reason, COUNT(*) AS record_count
FROM OPENAQ.DATA_QUALITY.REJECTED_RECORDS
GROUP BY rejection_reason
ORDER BY record_count DESC;

## Step 6: Staging - Clean Data with Interpolation

In [ ]:
%%sql -r create_staging
USE SCHEMA OPENAQ.STAGING;

CREATE OR REPLACE TABLE AIR_QUALITY_CLEAN AS
SELECT
    r.location_id,
    r.sensors_id,
    r.location,
    SPLIT_PART(r.location, '-', 1) AS city,
    r.datetime,
    DATE_TRUNC('day', r.datetime) AS measurement_date,
    DATE_TRUNC('month', r.datetime) AS measurement_month,
    EXTRACT(HOUR FROM r.datetime) AS measurement_hour,
    r.lat,
    r.lon,
    r.parameter,
    r.units,
    r.value,
    COALESCE(
        r.value,
        (LAG(r.value) OVER (PARTITION BY r.location_id, r.parameter ORDER BY r.datetime)
         + LEAD(r.value) OVER (PARTITION BY r.location_id, r.parameter ORDER BY r.datetime)) / 2
    ) AS interpolated_value
FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS r
WHERE r.value IS NOT NULL
  AND r.value >= 0
  AND r.datetime IS NOT NULL
  AND r.location IS NOT NULL AND r.location != ''
  AND r.parameter IS NOT NULL AND r.parameter != ''
  AND r.lat IS NOT NULL AND r.lon IS NOT NULL
  AND NOT EXISTS (
      SELECT 1 FROM OPENAQ.DATA_QUALITY.REJECTED_RECORDS rej
      WHERE rej.location_id = r.location_id
        AND rej.sensors_id = r.sensors_id
        AND rej.datetime = r.datetime
        AND rej.parameter = r.parameter
        AND rej.rejection_reason = 'Extreme outlier (>5 std dev from mean)'
  );

In [ ]:
%%sql -r staging_verify
-- Verify staging data
SELECT parameter, COUNT(*) AS records, ROUND(AVG(value), 3) AS avg_value
FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
GROUP BY parameter;

## Step 7: Data Mart Tables

### 7A: Monthly City Pollution (CO & SO2)

In [ ]:
%%sql -r create_monthly_pollution
USE SCHEMA OPENAQ.MART;

CREATE OR REPLACE TABLE MONTHLY_CITY_POLLUTION AS
SELECT
    city,
    location,
    measurement_month,
    parameter,
    units,
    AVG(interpolated_value) AS avg_monthly_value,
    COUNT(*) AS measurement_count,
    MIN(interpolated_value) AS min_value,
    MAX(interpolated_value) AS max_value
FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
WHERE parameter IN ('co', 'so2')
GROUP BY city, location, measurement_month, parameter, units;

### 7B: Daily City PM2.5

In [ ]:
%%sql -r create_daily_pm25
CREATE OR REPLACE TABLE DAILY_CITY_PM25 AS
SELECT
    city,
    location,
    measurement_date,
    AVG(interpolated_value) AS avg_daily_pm25,
    COUNT(*) AS measurement_count,
    MIN(interpolated_value) AS min_pm25,
    MAX(interpolated_value) AS max_pm25
FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
WHERE parameter = 'pm25'
GROUP BY city, location, measurement_date;

### 7C: Hourly City Pollution (PM2.5, CO, SO2)

In [ ]:
%%sql -r create_hourly_pollution
CREATE OR REPLACE TABLE HOURLY_CITY_POLLUTION AS
SELECT
    city,
    location,
    measurement_date,
    measurement_hour,
    parameter,
    units,
    AVG(interpolated_value) AS avg_hourly_value,
    COUNT(*) AS measurement_count
FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
WHERE parameter IN ('pm25', 'co', 'so2')
GROUP BY city, location, measurement_date, measurement_hour, parameter, units;

### 7D: Country Air Quality Index (High / Moderate / Low)

In [ ]:
%%sql -r create_aqi
CREATE OR REPLACE TABLE COUNTRY_AIR_QUALITY_INDEX AS
WITH hourly_params AS (
    SELECT
        city,
        location,
        measurement_date,
        measurement_hour,
        lat,
        lon,
        MAX(CASE WHEN parameter = 'pm25' THEN interpolated_value END) AS pm25_value,
        MAX(CASE WHEN parameter = 'pm10' THEN interpolated_value END) AS pm10_value,
        MAX(CASE WHEN parameter = 'so2' THEN interpolated_value END) AS so2_value,
        MAX(CASE WHEN parameter = 'co' THEN interpolated_value END) AS co_value
    FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
    WHERE parameter IN ('pm25', 'pm10', 'so2', 'co')
    GROUP BY city, location, measurement_date, measurement_hour, lat, lon
),
scored AS (
    SELECT
        *,
        COALESCE(pm25_value / 35.0, 0) * 0.4
        + COALESCE(pm10_value / 150.0, 0) * 0.2
        + COALESCE(so2_value / 0.075, 0) * 0.2
        + COALESCE(co_value / 9.0, 0) * 0.2 AS aqi_score
    FROM hourly_params
)
SELECT
    location,
    city,
    lat,
    lon,
    measurement_date,
    measurement_hour,
    pm25_value,
    pm10_value,
    so2_value,
    co_value,
    aqi_score,
    CASE
        WHEN aqi_score >= 1.0 THEN 'High'
        WHEN aqi_score >= 0.5 THEN 'Moderate'
        ELSE 'Low'
    END AS air_quality_level
FROM scored;

## Step 8: Analytic Queries (Business Requirements)

### Query 1: Monthly 90th Percentile CO & SO2 Cities
For any given month, find all cities with average monthly CO and SO2 levels in the 90th percentile globally.

In [ ]:
%%sql -r query1_90th_percentile
WITH monthly_stats AS (
    SELECT
        city,
        parameter,
        avg_monthly_value,
        PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY avg_monthly_value)
            OVER (PARTITION BY parameter, measurement_month) AS p90_threshold
    FROM OPENAQ.MART.MONTHLY_CITY_POLLUTION
    WHERE measurement_month = '2022-05-01'::DATE
)
SELECT DISTINCT
    city,
    parameter,
    avg_monthly_value,
    p90_threshold
FROM monthly_stats
WHERE avg_monthly_value >= p90_threshold
ORDER BY parameter, avg_monthly_value DESC;

### Query 2: Top 5 Daily PM2.5 Cities
For any given day, find the top 5 cities globally with the highest daily average PM2.5 levels.

In [ ]:
%%sql -r query2_top5_pm25
SELECT
    city,
    location,
    measurement_date,
    avg_daily_pm25,
    measurement_count
FROM OPENAQ.MART.DAILY_CITY_PM25
WHERE measurement_date = '2022-05-03'::DATE
ORDER BY avg_daily_pm25 DESC
LIMIT 5;

### Query 3: Top 10 Hourly PM2.5 Cities + CO/SO2 Mean, Median, Mode
For any given hour, find the top 10 cities with highest daily average PM2.5, plus mean/median/mode of CO & SO2.

In [ ]:
%%sql -r query3_top10_pm25_stats
WITH top10_pm25_cities AS (
    SELECT city, location, measurement_date, measurement_hour, avg_hourly_value AS pm25_avg
    FROM OPENAQ.MART.HOURLY_CITY_POLLUTION
    WHERE parameter = 'pm25'
      AND measurement_date = '2022-05-03'::DATE
      AND measurement_hour = 8
    ORDER BY pm25_avg DESC
    LIMIT 10
),
co_so2_daily AS (
    SELECT
        c.city,
        c.location,
        s.parameter,
        s.interpolated_value
    FROM top10_pm25_cities c
    JOIN OPENAQ.STAGING.AIR_QUALITY_CLEAN s
        ON c.city = s.city
        AND c.measurement_date = s.measurement_date
    WHERE s.parameter IN ('co', 'so2')
)
SELECT
    t.city,
    t.location,
    t.pm25_avg,
    d.parameter,
    AVG(d.interpolated_value) AS mean_value,
    MEDIAN(d.interpolated_value) AS median_value,
    MODE(d.interpolated_value) AS mode_value
FROM top10_pm25_cities t
LEFT JOIN co_so2_daily d
    ON t.city = d.city AND t.location = d.location
GROUP BY t.city, t.location, t.pm25_avg, d.parameter
ORDER BY t.pm25_avg DESC, d.parameter;

### Query 4: Hourly Air Quality Index by Country
For any given hour, report the air quality index per country with 3 levels: High, Moderate, Low.

In [ ]:
%%sql -r query4_aqi_by_location
SELECT
    location,
    city,
    lat,
    lon,
    measurement_date,
    measurement_hour,
    pm25_value,
    pm10_value,
    so2_value,
    co_value,
    ROUND(aqi_score, 3) AS aqi_score,
    air_quality_level
FROM OPENAQ.MART.COUNTRY_AIR_QUALITY_INDEX
WHERE measurement_date = '2022-05-03'::DATE
  AND measurement_hour = 8
ORDER BY aqi_score DESC;

## Data Quality Ad-Hoc Report

In [ ]:
%%sql -r dq_report
SELECT
    rejection_reason,
    parameter,
    COUNT(*) AS rejected_count,
    MIN(value) AS min_rejected_value,
    MAX(value) AS max_rejected_value,
    ROUND(AVG(value), 3) AS avg_rejected_value
FROM OPENAQ.DATA_QUALITY.REJECTED_RECORDS
GROUP BY rejection_reason, parameter
ORDER BY rejected_count DESC;

## Scalability: Loading Historical Data from OpenAQ S3
Uncomment and run the cell below to load historical data from the public S3 bucket.

In [ ]:
%%sql -r historical_load
-- To load historical 2017 data from S3 (uncomment to run):
-- COPY INTO OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS (
--     location_id, sensors_id, location, datetime, lat, lon, parameter, units, value
-- )
-- FROM @OPENAQ.RAW.OPENAQ_S3_STAGE/records/csv.gz/
-- PATTERN = '.*location.*2017.*\.csv\.gz'
-- FILE_FORMAT = (FORMAT_NAME = OPENAQ.RAW.CSV_FORMAT)
-- ON_ERROR = 'CONTINUE';

---
## Production Enhancements: TB-Scale & Pipeline Automation
The following sections add clustering keys, stored procedures with error handling, dynamic tables for auto-refresh, and task-based scheduling.

### ETL Schema & Pipeline Log

In [ ]:
%%sql -r create_etl_schema
CREATE SCHEMA IF NOT EXISTS OPENAQ.ETL
    COMMENT = 'Stored procedures and pipeline orchestration';

CREATE OR REPLACE TABLE OPENAQ.ETL.PIPELINE_LOG (
    run_id          VARCHAR(50),
    step_name       VARCHAR(200),
    status          VARCHAR(20),
    rows_affected   INTEGER,
    error_message   VARCHAR(5000),
    started_at      TIMESTAMP_NTZ,
    completed_at    TIMESTAMP_NTZ
)
COMMENT = 'ETL pipeline execution audit log';

### TB-Scale: Add Clustering Keys + Source Tracking

In [ ]:
%%sql -r add_clustering_raw
-- Add source tracking columns to raw table
ALTER TABLE OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS ADD COLUMN IF NOT EXISTS
    source_name VARCHAR(200) DEFAULT 'openaq_s3';

ALTER TABLE OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS ADD COLUMN IF NOT EXISTS
    source_file VARCHAR(1000);

-- Add clustering key for TB-scale query performance
ALTER TABLE OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS
    CLUSTER BY (DATE_TRUNC('month', datetime), parameter);

In [ ]:
%%sql -r add_clustering_staging
-- Add clustering to staging table for fast joins
ALTER TABLE OPENAQ.STAGING.AIR_QUALITY_CLEAN
    CLUSTER BY (measurement_date, parameter, city);

In [ ]:
%%sql -r add_source_dq
-- Add source column to rejected records
ALTER TABLE OPENAQ.DATA_QUALITY.REJECTED_RECORDS ADD COLUMN IF NOT EXISTS
    source_name VARCHAR(200);

### Stored Procedures: Production ETL Pipeline

In [ ]:
%%sql -r sp_data_quality
-- Data Quality Procedure: identifies and logs rejected records
CREATE OR REPLACE PROCEDURE OPENAQ.ETL.SP_RUN_DATA_QUALITY(P_RUN_ID VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET v_start TIMESTAMP_NTZ := CURRENT_TIMESTAMP();
    LET v_rows INTEGER := 0;

    DELETE FROM OPENAQ.DATA_QUALITY.REJECTED_RECORDS
    WHERE rejected_at < DATEADD('day', -30, CURRENT_TIMESTAMP());

    INSERT INTO OPENAQ.DATA_QUALITY.REJECTED_RECORDS
        (location_id, sensors_id, location, datetime, lat, lon, parameter, units, value, rejection_reason, source_name)
    SELECT location_id, sensors_id, location, datetime, lat, lon, parameter, units, value,
        CASE
            WHEN value IS NULL THEN 'NULL measurement value'
            WHEN datetime IS NULL THEN 'NULL datetime'
            WHEN location IS NULL OR location = '' THEN 'NULL or empty location'
            WHEN parameter IS NULL OR parameter = '' THEN 'NULL or empty parameter'
            WHEN lat IS NULL OR lon IS NULL THEN 'NULL coordinates'
        END,
        source_name
    FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS
    WHERE value IS NULL
       OR datetime IS NULL
       OR location IS NULL OR location = ''
       OR parameter IS NULL OR parameter = ''
       OR lat IS NULL OR lon IS NULL;

    v_rows := SQLROWCOUNT;

    INSERT INTO OPENAQ.DATA_QUALITY.REJECTED_RECORDS
        (location_id, sensors_id, location, datetime, lat, lon, parameter, units, value, rejection_reason, source_name)
    SELECT location_id, sensors_id, location, datetime, lat, lon, parameter, units, value,
        'Negative measurement value', source_name
    FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS
    WHERE value < 0;

    v_rows := :v_rows + SQLROWCOUNT;

    INSERT INTO OPENAQ.DATA_QUALITY.REJECTED_RECORDS
        (location_id, sensors_id, location, datetime, lat, lon, parameter, units, value, rejection_reason, source_name)
    SELECT r.location_id, r.sensors_id, r.location, r.datetime, r.lat, r.lon, r.parameter, r.units, r.value,
        'Extreme outlier (>5 std dev from mean)', r.source_name
    FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS r
    JOIN (
        SELECT parameter, AVG(value) AS avg_val, STDDEV(value) AS std_val
        FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS
        WHERE value IS NOT NULL AND value >= 0
        GROUP BY parameter
    ) stats ON r.parameter = stats.parameter
    WHERE r.value > (stats.avg_val + 5 * stats.std_val)
      AND stats.std_val > 0;

    v_rows := :v_rows + SQLROWCOUNT;

    INSERT INTO OPENAQ.ETL.PIPELINE_LOG (run_id, step_name, status, rows_affected, started_at, completed_at)
    VALUES (:P_RUN_ID, 'DATA_QUALITY', 'SUCCESS', :v_rows, :v_start, CURRENT_TIMESTAMP());

    RETURN 'Data quality complete. Rejected records: ' || :v_rows::VARCHAR;

EXCEPTION
    WHEN OTHER THEN
        INSERT INTO OPENAQ.ETL.PIPELINE_LOG (run_id, step_name, status, error_message, started_at, completed_at)
        VALUES (:P_RUN_ID, 'DATA_QUALITY', 'FAILED', SQLERRM, :v_start, CURRENT_TIMESTAMP());
        RETURN 'FAILED: ' || SQLERRM;
END;

In [ ]:
%%sql -r sp_staging_refresh
-- Staging Refresh Procedure: rebuilds clean data with interpolation
CREATE OR REPLACE PROCEDURE OPENAQ.ETL.SP_REFRESH_STAGING(P_RUN_ID VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET v_start TIMESTAMP_NTZ := CURRENT_TIMESTAMP();
    LET v_rows INTEGER := 0;

    CREATE OR REPLACE TABLE OPENAQ.STAGING.AIR_QUALITY_CLEAN AS
    SELECT
        r.location_id,
        r.sensors_id,
        r.location,
        SPLIT_PART(r.location, '-', 1) AS city,
        r.datetime,
        DATE_TRUNC('day', r.datetime)::DATE AS measurement_date,
        DATE_TRUNC('month', r.datetime)::DATE AS measurement_month,
        EXTRACT(HOUR FROM r.datetime)::INTEGER AS measurement_hour,
        r.lat,
        r.lon,
        r.parameter,
        r.units,
        r.value,
        COALESCE(
            r.value,
            (LAG(r.value) OVER (PARTITION BY r.location_id, r.parameter ORDER BY r.datetime)
             + LEAD(r.value) OVER (PARTITION BY r.location_id, r.parameter ORDER BY r.datetime)) / 2
        ) AS interpolated_value
    FROM OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS r
    WHERE r.value IS NOT NULL
      AND r.value >= 0
      AND r.datetime IS NOT NULL
      AND r.location IS NOT NULL AND r.location != ''
      AND r.parameter IS NOT NULL AND r.parameter != ''
      AND r.lat IS NOT NULL AND r.lon IS NOT NULL
      AND NOT EXISTS (
          SELECT 1 FROM OPENAQ.DATA_QUALITY.REJECTED_RECORDS rej
          WHERE rej.location_id = r.location_id
            AND rej.sensors_id = r.sensors_id
            AND rej.datetime = r.datetime
            AND rej.parameter = r.parameter
            AND rej.rejection_reason = 'Extreme outlier (>5 std dev from mean)'
      );

    SELECT COUNT(*) INTO :v_rows FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN;

    ALTER TABLE OPENAQ.STAGING.AIR_QUALITY_CLEAN
        CLUSTER BY (measurement_date, parameter, city);

    INSERT INTO OPENAQ.ETL.PIPELINE_LOG (run_id, step_name, status, rows_affected, started_at, completed_at)
    VALUES (:P_RUN_ID, 'STAGING_REFRESH', 'SUCCESS', :v_rows, :v_start, CURRENT_TIMESTAMP());

    RETURN 'Staging refresh complete. Clean records: ' || :v_rows::VARCHAR;

EXCEPTION
    WHEN OTHER THEN
        INSERT INTO OPENAQ.ETL.PIPELINE_LOG (run_id, step_name, status, error_message, started_at, completed_at)
        VALUES (:P_RUN_ID, 'STAGING_REFRESH', 'FAILED', SQLERRM, :v_start, CURRENT_TIMESTAMP());
        RETURN 'FAILED: ' || SQLERRM;
END;

In [ ]:
%%sql -r sp_s3_load
-- Incremental Load from S3 by year/month pattern
CREATE OR REPLACE PROCEDURE OPENAQ.ETL.SP_LOAD_FROM_S3(P_RUN_ID VARCHAR, P_YEAR VARCHAR, P_MONTH VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET v_start TIMESTAMP_NTZ := CURRENT_TIMESTAMP();
    LET v_pattern VARCHAR := '.*locationId=.*/' || P_YEAR || '/' || P_MONTH || '/.*\.csv\.gz';

    COPY INTO OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS (
        location_id, sensors_id, location, datetime, lat, lon, parameter, units, value
    )
    FROM @OPENAQ.RAW.OPENAQ_S3_STAGE/records/csv.gz/
    PATTERN = :v_pattern
    FILE_FORMAT = (TYPE='CSV' SKIP_HEADER=1 FIELD_OPTIONALLY_ENCLOSED_BY='"' NULL_IF=('','NA','null') COMPRESSION='GZIP')
    ON_ERROR = 'CONTINUE';

    LET v_rows INTEGER := SQLROWCOUNT;

    UPDATE OPENAQ.RAW.AIR_QUALITY_MEASUREMENTS
    SET source_name = 'openaq_s3_' || :P_YEAR || '_' || :P_MONTH
    WHERE source_name = 'openaq_s3'
      AND source_file IS NULL
      AND ingested_at >= :v_start;

    INSERT INTO OPENAQ.ETL.PIPELINE_LOG (run_id, step_name, status, rows_affected, started_at, completed_at)
    VALUES (:P_RUN_ID, 'S3_LOAD_' || :P_YEAR || '_' || :P_MONTH, 'SUCCESS', :v_rows, :v_start, CURRENT_TIMESTAMP());

    RETURN 'S3 load complete for ' || :P_YEAR || '-' || :P_MONTH || '. Rows: ' || :v_rows::VARCHAR;

EXCEPTION
    WHEN OTHER THEN
        INSERT INTO OPENAQ.ETL.PIPELINE_LOG (run_id, step_name, status, error_message, started_at, completed_at)
        VALUES (:P_RUN_ID, 'S3_LOAD_' || :P_YEAR || '_' || :P_MONTH, 'FAILED', SQLERRM, :v_start, CURRENT_TIMESTAMP());
        RETURN 'FAILED: ' || SQLERRM;
END;

In [ ]:
%%sql -r sp_orchestrator
-- Master Orchestrator: runs the full pipeline end-to-end
CREATE OR REPLACE PROCEDURE OPENAQ.ETL.SP_RUN_FULL_PIPELINE()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    LET v_run_id VARCHAR := 'RUN_' || TO_VARCHAR(CURRENT_TIMESTAMP(), 'YYYYMMDD_HH24MISS');
    LET v_result VARCHAR;

    CALL OPENAQ.ETL.SP_RUN_DATA_QUALITY(:v_run_id) INTO :v_result;
    IF (STARTSWITH(:v_result, 'FAILED')) THEN
        RETURN 'Pipeline aborted at DATA_QUALITY: ' || :v_result;
    END IF;

    CALL OPENAQ.ETL.SP_REFRESH_STAGING(:v_run_id) INTO :v_result;
    IF (STARTSWITH(:v_result, 'FAILED')) THEN
        RETURN 'Pipeline aborted at STAGING_REFRESH: ' || :v_result;
    END IF;

    -- Dynamic tables auto-refresh from staging (no manual mart rebuild needed)
    RETURN 'Pipeline complete. Run ID: ' || :v_run_id;
END;

### Dynamic Tables: Auto-Refreshing Mart (replaces static CTAS tables)

In [ ]:
%%sql -r dt_monthly
-- Monthly City Pollution - auto-refreshes when staging changes
CREATE OR REPLACE DYNAMIC TABLE OPENAQ.MART.MONTHLY_CITY_POLLUTION
    TARGET_LAG = '1 hour'
    WAREHOUSE = COMPUTE_WH
AS
SELECT
    city,
    location,
    measurement_month,
    parameter,
    units,
    AVG(interpolated_value) AS avg_monthly_value,
    COUNT(*) AS measurement_count,
    MIN(interpolated_value) AS min_value,
    MAX(interpolated_value) AS max_value
FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
WHERE parameter IN ('co', 'so2')
GROUP BY city, location, measurement_month, parameter, units;

In [ ]:
%%sql -r dt_daily_pm25
-- Daily City PM2.5 - auto-refreshes when staging changes
CREATE OR REPLACE DYNAMIC TABLE OPENAQ.MART.DAILY_CITY_PM25
    TARGET_LAG = '1 hour'
    WAREHOUSE = COMPUTE_WH
AS
SELECT
    city,
    location,
    measurement_date,
    AVG(interpolated_value) AS avg_daily_pm25,
    COUNT(*) AS measurement_count,
    MIN(interpolated_value) AS min_pm25,
    MAX(interpolated_value) AS max_pm25
FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
WHERE parameter = 'pm25'
GROUP BY city, location, measurement_date;

In [ ]:
%%sql -r dt_hourly
-- Hourly City Pollution - auto-refreshes when staging changes
CREATE OR REPLACE DYNAMIC TABLE OPENAQ.MART.HOURLY_CITY_POLLUTION
    TARGET_LAG = '1 hour'
    WAREHOUSE = COMPUTE_WH
AS
SELECT
    city,
    location,
    measurement_date,
    measurement_hour,
    parameter,
    units,
    AVG(interpolated_value) AS avg_hourly_value,
    COUNT(*) AS measurement_count
FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
WHERE parameter IN ('pm25', 'co', 'so2')
GROUP BY city, location, measurement_date, measurement_hour, parameter, units;

In [ ]:
%%sql -r dt_aqi
-- Country Air Quality Index - auto-refreshes when staging changes
CREATE OR REPLACE DYNAMIC TABLE OPENAQ.MART.COUNTRY_AIR_QUALITY_INDEX
    TARGET_LAG = '1 hour'
    WAREHOUSE = COMPUTE_WH
AS
WITH hourly_params AS (
    SELECT
        city,
        location,
        measurement_date,
        measurement_hour,
        lat,
        lon,
        MAX(CASE WHEN parameter = 'pm25' THEN interpolated_value END) AS pm25_value,
        MAX(CASE WHEN parameter = 'pm10' THEN interpolated_value END) AS pm10_value,
        MAX(CASE WHEN parameter = 'so2' THEN interpolated_value END) AS so2_value,
        MAX(CASE WHEN parameter = 'co' THEN interpolated_value END) AS co_value
    FROM OPENAQ.STAGING.AIR_QUALITY_CLEAN
    WHERE parameter IN ('pm25', 'pm10', 'so2', 'co')
    GROUP BY city, location, measurement_date, measurement_hour, lat, lon
),
scored AS (
    SELECT
        *,
        COALESCE(pm25_value / 35.0, 0) * 0.4
        + COALESCE(pm10_value / 150.0, 0) * 0.2
        + COALESCE(so2_value / 0.075, 0) * 0.2
        + COALESCE(co_value / 9.0, 0) * 0.2 AS aqi_score
    FROM hourly_params
)
SELECT
    location,
    city,
    lat,
    lon,
    measurement_date,
    measurement_hour,
    pm25_value,
    pm10_value,
    so2_value,
    co_value,
    aqi_score,
    CASE
        WHEN aqi_score >= 1.0 THEN 'High'
        WHEN aqi_score >= 0.5 THEN 'Moderate'
        ELSE 'Low'
    END AS air_quality_level
FROM scored;

### Task-Based Scheduling (Automated Pipeline)

In [ ]:
%%sql -r create_task_pipeline
-- Root task: runs the full pipeline every 6 hours
CREATE OR REPLACE TASK OPENAQ.ETL.TASK_FULL_PIPELINE
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = 'USING CRON 0 */6 * * * America/Denver'
    COMMENT = 'Runs full ETL pipeline every 6 hours'
AS
    CALL OPENAQ.ETL.SP_RUN_FULL_PIPELINE();

In [ ]:
%%sql -r create_task_monitor
-- Monitoring task: checks for pipeline failures every hour
CREATE OR REPLACE TASK OPENAQ.ETL.TASK_MONITOR_PIPELINE
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = 'USING CRON 0 * * * * America/Denver'
    COMMENT = 'Monitors pipeline health and logs failures'
AS
    INSERT INTO OPENAQ.ETL.PIPELINE_LOG (run_id, step_name, status, rows_affected, started_at, completed_at)
    SELECT
        'MONITOR_' || TO_VARCHAR(CURRENT_TIMESTAMP(), 'YYYYMMDD_HH24MISS'),
        'HEALTH_CHECK',
        CASE WHEN COUNT_IF(status = 'FAILED') > 0 THEN 'ALERT' ELSE 'HEALTHY' END,
        COUNT_IF(status = 'FAILED'),
        MIN(started_at),
        CURRENT_TIMESTAMP()
    FROM OPENAQ.ETL.PIPELINE_LOG
    WHERE started_at >= DATEADD('hour', -6, CURRENT_TIMESTAMP());

In [ ]:
%%sql -r show_tasks
-- Enable tasks (uncomment to activate in production)
-- ALTER TASK OPENAQ.ETL.TASK_FULL_PIPELINE RESUME;
-- ALTER TASK OPENAQ.ETL.TASK_MONITOR_PIPELINE RESUME;

-- Check task status
SHOW TASKS IN SCHEMA OPENAQ.ETL;

### Run Full Pipeline & Verify

In [ ]:
%%sql -r run_pipeline
-- Execute the full production pipeline
CALL OPENAQ.ETL.SP_RUN_FULL_PIPELINE();

In [ ]:
%%sql -r pipeline_log
-- Verify pipeline execution log
SELECT * FROM OPENAQ.ETL.PIPELINE_LOG ORDER BY started_at DESC;

### TB-Scale: Load Historical 2017 Data from S3
Uncomment and run to load all 12 months of 2017 data.

In [ ]:
%%sql -r historical_load
-- Uncomment to load historical 2017 data month-by-month from S3:
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '01');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '02');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '03');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '04');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '05');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '06');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '07');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '08');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '09');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '10');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '11');
-- CALL OPENAQ.ETL.SP_LOAD_FROM_S3('HISTORICAL_2017', '2017', '12');
-- Then run: CALL OPENAQ.ETL.SP_RUN_FULL_PIPELINE();